# Vargas signature validation

**Note**:
>> Functions from /fun taken from Vargas repo
>> Requires log-transformed, scaled, normalized (TPM) expression data 

In [ ]:
import pandas as pd
import numpy as np
import os,pickle
from sklearn.preprocessing import StandardScaler
from fun import predictive_model, validation 
from fun import utilities as ut
%matplotlib inline

np.random.seed(30)

### Import data

In [ ]:
_w0=pd.read_csv('.../_w0_counts.csv')
_w1=pd.read_csv('.../_w1_counts.csv')
_w2=pd.read_csv('.../_w2_counts.csv')
_w9=pd.read_csv('.../_w9_counts.csv')
_w14=pd.read_csv('.../_w14_counts.csv')
_w26=pd.read_csv('.../_w26_counts.csv')

In [ ]:
### Import model - fs3
model_ob=pickle.load(open('.../TBRecur/vargas_final-ML-model/opt_score_rocauc/ATB_nonATB_combine_fs_3_random_forest.pickle', 'rb')) ##rb - read mod; wb ==write mode)
reduced_model=model_ob[1]

In [5]:
features=model_ob[0]

### Preprocess count data
> Using TPM count matrix per timepoint 

- log2 transform
- standard scaler

#### Counts

#### Log transform

In [ ]:
# logtransform (log2)
dfs = [_w0,_w1,_w2,_w9,_w14,_w26]
df_log = {}
for idx,df in enumerate(dfs):
    df_log[f'df_{idx}'] = np.log2(df+1) ## same as Vargas code

####  Standard scalar (per timepoint)

In [19]:
# loop through for all log-transformed dfs
scaled = {}
for key,df in df_log.items():
    d_s = StandardScaler().fit_transform(df)
    d_s = pd.DataFrame(data=d_s, index=df.index, columns=df.columns).reset_index()
    d_s.drop(columns=['gene_id'], inplace=True)
    d_st = d_s.transpose()
    d_st.columns=d_st.iloc[0]
    d_st.drop(labels='symbol', axis=0, inplace=True)
    d_st=d_st.astype('float64') # int64 made it weird
    scaled[key]=d_st

### Get predictions

In [20]:
## Get dfs with feature ratios
ratios={}
for key, df in scaled.items():
    rat = {}
    for pair in features:
        gene1, gene2 = pair.split('_')  
        rat[pair]=df[gene1] / df[gene2]
    ratios[key]=pd.DataFrame(rat)

In [21]:
### get scores from model
scores={}
for key, df in ratios.items():
    pred=reduced_model.predict(df)
    scores[key] = pd.DataFrame(pred, index=df.index, 
                         columns=[f'{key}_scores'])

In [ ]:
## write out indiviudal values
for key, df in scores.items():
    df.to_csv(os.path.join('.../', f'{key}_Vargas_scores.csv'),float_format="%.6f")    